# KadiPy - Module `kidas` : Documentation complète

> **kidas** (*Kadi Data Acquisition & Standardization*) est le module de manipulation de données du projet KadiPy.
> Il est conçu pour les besoins spécifiques de l'AgriTech en Afrique de l'Ouest (Bénin).

---

## Architecture générale

```
kadi.kidas
├── sources/
│   ├── base.py          → DataSource (classe abstraite)
│   ├── csv_source.py    → CSVDataSource
│   ├── excel_source.py  → ExcelDataSource
│   ├── json_source.py   → JSONDataSource
│   ├── netcdf_source.py → NetCDFDataSource
│   └── api_source.py    → APIDataSource
├── cleaner.py           → DataCleaner
├── validator.py         → DataValidator
├── normalizer.py        → DataNormalizer
├── cache.py             → DataCache
└── pipeline.py          → DataPipeline (point d'entrée)
```

## Table des notebooks à créer

| # | Notebook | Classe | Description |
|---|---|---|---|
| 01 | [01_csv_source.ipynb](./01_csv_source.ipynb) | `CSVDataSource` | CSV : encodage auto, délimiteur auto, fallbacks |
| 02 | [02_excel_source.ipynb](./02_excel_source.ipynb) | `ExcelDataSource` | Excel : cellules fusionnées, multi-feuilles, en-tête auto |
| 03 | [03_json_source.ipynb](./03_json_source.ipynb) | `JSONDataSource` | JSON : aplatissement imbriqué, dict ou fichier |
| 04 | [04_netcdf_source.ipynb](./04_netcdf_source.ipynb) | `NetCDFDataSource` | NetCDF : CHIRPS/ERA5, bbox Bénin, Dask |
| 05 | [05_api_source.ipynb](./05_api_source.ipynb) | `APIDataSource` | API REST : backoff exponentiel, rate limiting |
| 06 | [06_data_cleaner.ipynb](./06_data_cleaner.ipynb) | `DataCleaner` | Nettoyage : doublons, NaN, outliers, dates, texte |
| 07 | [07_data_validator.ipynb](./07_data_validator.ipynb) | `DataValidator` | Validation : schéma, plages, GPS, unicité, score qualité |
| 08 | [08_data_normalizer.ipynb](./08_data_normalizer.ipynb) | `DataNormalizer` | Normalisation : snake_case, FAO, marchés Bénin, unités |
| 09 | [09_data_cache.ipynb](./09_data_cache.ipynb) | `DataCache` | Cache SQLite : save/load, historique, expiration |
| 10 | [10_data_pipeline.ipynb](./10_data_pipeline.ipynb) | `DataPipeline` | Orchestrateur : API fluide, pipeline complet |

---

## Démarrage rapide

In [ ]:
# Installation des dépendances (si nécessaire)
# !pip install pandas openpyxl chardet scipy shapely Unidecode xarray requests

In [2]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import kadi.kidas as kidas
print(f"kidas version : {kidas.__version__}")
print(f"Classes disponibles : {kidas.__all__}")

kidas version : 1.0.0
Classes disponibles : ['CSVDataSource', 'ExcelDataSource', 'JSONDataSource', 'NetCDFDataSource', 'APIDataSource', 'DataCleaner', 'DataValidator', 'DataNormalizer', 'DataCache', 'DataPipeline', 'load_and_clean']


#### Explication de la sortie — Cellule 1 : Importation et vérification du module

Cette cellule vérifie que le module `kidas` est correctement installé et accessible depuis le chemin projet.

| Ligne affichée | Signification |
|---|---|
| `kidas version : 1.0.0` | La version courante du module `kidas` est **1.0.0**, définie dans `kidas.__version__`. |
| `Classes disponibles : [...]` | La liste `kidas.__all__` expose les **11 symboles publics** du module : les 5 sources de données (`CSVDataSource`, `ExcelDataSource`, `JSONDataSource`, `NetCDFDataSource`, `APIDataSource`), les 4 composants de traitement (`DataCleaner`, `DataValidator`, `DataNormalizer`, `DataCache`), l'orchestrateur (`DataPipeline`) et la fonction utilitaire (`load_and_clean`). |

> **En résumé :** Si cette cellule s'exécute sans erreur d'import, l'environnement est correctement configuré et tous les composants de `kidas` sont disponibles.

### Exemple minimal : charger un fichier CSV

In [3]:
import os
import tempfile
import pandas as pd
import numpy as np

# Création d'un fichier de démonstration
df_test = pd.DataFrame({
    'culture': ['Maïs', 'Niébé', 'igname', 'MAÏS', None],
    'prix_xof': [350, 500, 250, 350, np.nan],
    'rendement_kg': [1500.0, 800.0, 2000.0, 1500.0, np.nan],
})

fichier = os.path.join(tempfile.gettempdir(), 'demo_index.csv')
df_test.to_csv(fichier, index=False)

# Chargement et nettoyage en une ligne
df_propre, rapport = kidas.load_and_clean(fichier, cache=False)

print(f"Avant : {len(df_test)} lignes | Après : {len(df_propre)} lignes")
df_propre

Avant : 5 lignes | Après : 5 lignes


,culture,prix_xof,rendement_kg
0,Maïs,350.0,1500.0
1,Niébé,500.0,800.0
2,igname,250.0,2000.0
3,MAÏS,350.0,1500.0
4,NaN,362.5,1450.0


#### Explication de la sortie — Cellule 2 : Chargement CSV minimal (`load_and_clean`)

**Contexte :** Un DataFrame de démonstration est créé avec 5 lignes intentionnellement imparfaites
(doublons de casse, valeur `None`, `NaN`), sauvegardé en CSV temporaire, puis rechargé via `load_and_clean`.

**Ligne texte affichée :**
```
Avant : 5 lignes | Après : 5 lignes
```
Le nombre de lignes reste **5** car `load_and_clean` avec les paramètres par défaut applique
un nettoyage conservateur : les valeurs manquantes **numériques** sont imputées à la moyenne
de leur colonne, sans supprimer aucune ligne.

**Tableau affiché — analyse ligne par ligne :**

| index | culture | prix\_xof | rendement\_kg | Explication |
|---|---|---|---|---|
| 0 | `Maïs` | 350.0 | 1500.0 | Ligne originale, inchangée. |
| 1 | `Niébé` | 500.0 | 800.0 | Ligne originale, inchangée. |
| 2 | `igname` | 250.0 | 2000.0 | Casse originale conservée — pas de normalisation de texte à ce stade. |
| 3 | `MAÏS` | 350.0 | 1500.0 | Doublon de `Maïs` (casse différente) **conservé** — `remove_duplicates` est sensible à la casse par défaut. |
| 4 | `NaN` | 362.5 | 1450.0 | `culture` est `None` → reste `NaN` (texte non imputable). `prix_xof` et `rendement_kg`, initialement `NaN`, sont **imputés à la moyenne** des 4 autres valeurs : (350+500+250+350)/4 = **362.5** et (1500+800+2000+1500)/4 = **1450.0**. |

> **À retenir :** `load_and_clean` est un raccourci pratique pour un nettoyage de base.
> Pour un contrôle fin (suppression de doublons par casse, normalisation du texte, mapping FAO…),
> utiliser le pipeline complet avec `DataPipeline` comme montré dans l'exemple suivant.

### Exemple avancé : pipeline personnalisé complet

In [6]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

from kadi.kidas import DataPipeline, DataCache

cache_dir = os.path.join(tempfile.gettempdir(), 'kidas_index_cache')

df_final, rapport = (
    DataPipeline()
    .__class__()
    ._DataPipeline__class_init()  # voir ci-dessous : syntaxe correcte
)

# Syntaxe correcte :
pipeline = DataPipeline()
pipeline._cache = DataCache(cache_dir=cache_dir)

df_final, rapport = (
    pipeline
    .load_data(fichier)
    .add_cleaning_step('remove_duplicates')
    .add_cleaning_step('handle_missing_values', strategy='mean')
    .add_cleaning_step('standardize_text', columns=['culture'], case='lower')
    .add_validation_step({'culture': 'str', 'prix_xof': 'float'})
    .add_normalization_step({'crops': 'culture'})
    .execute(cache=False)
)

print(f"Score qualité : {rapport.get('quality_score', {}).get('overall', 'N/A')}")
print(f"Étapes appliquées : {rapport['etapes_appliquees']}")
df_final

AttributeError: 'DataPipeline' object has no attribute '_DataPipeline__class_init'

#### Explication de la sortie — Cellule 3 : Erreur `AttributeError` (code intentionnellement incorrect)

Cette cellule illustre une **erreur volontaire** pour montrer ce qu'il ne faut **pas** faire.

**Erreur obtenue :**
```
AttributeError: 'DataPipeline' object has no attribute '_DataPipeline__class_init'
```

**Cause détaillée :**

| Problème | Explication |
|---|---|
| `DataPipeline().__class__()` | Appelle la classe elle-même comme constructeur une seconde fois via `__class__` — usage incorrect. |
| `._DataPipeline__class_init()` | Tente d'accéder à une méthode privée Python avec *name mangling* (`__class_init` → `_DataPipeline__class_init`). Cette méthode **n'existe pas** dans `DataPipeline`. |

**Ligne déclenchante (traceback ligne 13) :**
```python
DataPipeline().__class__()._DataPipeline__class_init()  # ← INCORRECT
```

> **Leçon :** Ne jamais appeler directement les méthodes privées manglées (`__méthode`).
> L'API publique de `DataPipeline` s'utilise via les méthodes chaînables `.load_data()`,
> `.add_cleaning_step()`, etc. La cellule suivante montre la syntaxe correcte.

In [7]:
# Version corrigée (sans l'artefact ci-dessus)
pipeline_propre = DataPipeline()
pipeline_propre._cache = DataCache(cache_dir=cache_dir)

df_final, rapport = (
    pipeline_propre
    .load_data(fichier)
    .add_cleaning_step('remove_duplicates')
    .add_cleaning_step('handle_missing_values', strategy='mean')
    .add_cleaning_step('standardize_text', columns=['culture'], case='lower')
    .add_validation_step({'culture': 'str', 'prix_xof': 'float'})
    .add_normalization_step({'crops': 'culture'})
    .execute(cache=False)
)

score = rapport.get('quality_score', {})
print(f"Score qualité global : {score.get('overall', 'N/A')}")
df_final

Score qualité global : 0.9033


,culture,prix_xof,rendement_kg
0,maize,350.0,1500.0
1,cowpea,500.0,800.0
2,yam,250.0,2000.0
3,maize,350.0,1500.0
4,NaN,362.5,1450.0


#### Explication de la sortie — Cellule 4 : Pipeline complet corrigé

Cette cellule exécute le pipeline `DataPipeline` avec l'API fluide **correcte**, en enchaînant
nettoyage, validation et normalisation.

**Étapes appliquées dans l'ordre :**

| Ordre | Méthode appelée | Effet sur les données |
|---|---|---|
| 1 | `load_data(fichier)` | Charge le CSV (5 lignes brutes). |
| 2 | `add_cleaning_step('remove_duplicates')` | Supprime les lignes **strictement identiques** sur toutes les colonnes. Ici aucune n'est supprimée : `Maïs` et `MAÏS` diffèrent par la casse. |
| 3 | `add_cleaning_step('handle_missing_values', strategy='mean')` | Remplace les `NaN` **numériques** par la moyenne de la colonne. `prix_xof` manquant → 362.5, `rendement_kg` manquant → 1450.0. |
| 4 | `add_cleaning_step('standardize_text', columns=['culture'], case='lower')` | Met la colonne `culture` **entièrement en minuscules** (`Maïs` → `maïs`, `MAÏS` → `maïs`, `Niébé` → `niébé`). |
| 5 | `add_validation_step({'culture': 'str', 'prix_xof': 'float'})` | Vérifie la conformité des types ; les résultats alimentent le **score qualité**. |
| 6 | `add_normalization_step({'crops': 'culture'})` | Traduit les noms de cultures vers la **nomenclature FAO** en anglais : `maïs` → `maize`, `niébé` → `cowpea`, `igname` → `yam`. |
| 7 | `execute(cache=False)` | Lance l'exécution complète ; retourne le tuple `(df_final, rapport)`. |

**Ligne texte affichée :**
```
Score qualité global : 0.9033
```
Le score de **0.9033 / 1.0** reflète une qualité élevée après traitement. Il est calculé par
`DataValidator` sur la base de la complétude, de la conformité des types et de l'absence
de valeurs aberrantes. Il n'atteint pas 1.0 car la ligne 4 conserve encore `NaN` dans `culture`
(un champ texte `None` ne peut pas être imputé automatiquement).

**Tableau final affiché — analyse ligne par ligne :**

| index | culture | prix\_xof | rendement\_kg | Transformation appliquée |
|---|---|---|---|---|
| 0 | `maize` | 350.0 | 1500.0 | `Maïs` → minuscules puis FAO (`maize`). |
| 1 | `cowpea` | 500.0 | 800.0 | `Niébé` → minuscules puis FAO (`cowpea` = niébé = pois à vaches). |
| 2 | `yam` | 250.0 | 2000.0 | `igname` → FAO (`yam`). |
| 3 | `maize` | 350.0 | 1500.0 | `MAÏS` → minuscules puis FAO (`maize`) — doublon de la ligne 0, conservé. |
| 4 | `NaN` | 362.5 | 1450.0 | `culture` reste `NaN` (non imputable). Numériques imputés à la moyenne des 4 autres lignes. |

> **À retenir :** L'API fluide de `DataPipeline` est le mode d'utilisation recommandé pour
> tout traitement avancé. Le dictionnaire `rapport` retourné contient le score qualité détaillé
> et la liste des étapes appliquées, utile pour l'audit et la traçabilité.

---

## Flux de données complet

```
Fichier CSV / Excel / JSON / NetCDF   API REST
          ↓                              ↓
     DataSource (lecture + métadonnées)
          ↓
     DataCleaner (doublons, NaN, outliers, dates, texte)
          ↓
     DataValidator (schéma, plages, GPS, unicité → score qualité)
          ↓
     DataNormalizer (snake_case, FAO, marchés Bénin, unités)
          ↓
     DataCache (SQLite, compression, versioning)
          ↓
     pd.DataFrame prêt à l'emploi + rapport complet
```

## Exceptions

Toutes les exceptions sont centralisées dans `kadi/exceptions.py` :

| Exception | Levée par |
|---|---|
| `KidasConnectionError` | Fichier/API inaccessible |
| `KidasReadError` | Erreur de lecture/décodage |
| `KidasWriteError` | Erreur d'écriture |
| `KidasCleaningError` | Méthode ou paramètre invalide |
| `KidasValidationError` | Données invalides |
| `KidasCacheError` | Problème de base SQLite |
| `KidasPipelineError` | Pipeline mal configuré |